# ICE-NAV iceberg trajectory experiment

This notebook reproduces the Milestone 7 track-only Random Forest experiment. The source is the BYU/NIC Consolidated Antarctic Iceberg Tracking Database v8.0. Validation is daily (+24 h) and applies to giant tabular icebergs, not the smaller synthetic demo contacts.

## Leakage audit

| Feature | Latest permitted timestamp | Audit |
|---|---|---|
| latitude / longitude | T0 | current observation only |
| current velocity | T0 | computed from T-1 to T0 |
| previous velocity | T-1 | computed from T-2 to T-1 |
| prior interval | T0 | timestamps no later than T0 |
| horizon | T0 | requested forecast horizon |
| day of year | T0 | derived from T0 |

The target position is used only after prediction for supervised fitting or evaluation. Group holdout keeps iceberg IDs disjoint; temporal holdout trains on earlier prediction times and tests on later ones.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
sys.path.insert(0, str(root))
from backend.ingestion.train_iceberg_model import load_tracks
from backend.engine.iceberg_features import build_examples

tracks = load_tracks(root / 'backend/data/historical/iceberg_tracks.csv')
examples = [example for track in tracks.values() for example in build_examples(track, 24)]
assert all(example['latest_feature_timestamp'] <= example['prediction_time'] < example['target_timestamp'] for example in examples)
len(tracks), len(examples)

## Reproduce model and holdout metrics

The command below rebuilds both the model and metadata with a fixed random seed. Hyperparameters are fixed before holdout evaluation; neither holdout set is used for tuning.

In [ ]:
from backend.ingestion.train_iceberg_model import main
main()